# Bayesian LoRA with Kronecker-Factored Laplace Approximation

This notebook demonstrates uncertainty quantification using Bayesian LoRA.

**Requirements:**
- GPU runtime (Runtime > Change runtime type > GPU)
- ~15GB GPU memory (works on free Colab T4)

**What this does:**
1. Computes Kronecker factors from training data
2. Uses Laplace approximation to get posterior over LoRA parameters
3. Samples from predictive distribution to compute uncertainty metrics

## 1. Setup Environment

In [ ]:
# Clone repository (if not already cloned)
import os
if not os.path.exists('adv-attacks-blllm'):
    !git clone https://github.com/YOUR_USERNAME/adv-attacks-blllm.git
    %cd adv-attacks-blllm
else:
    %cd adv-attacks-blllm
    !git pull  # Update to latest version

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft torch accelerate bayesian-lora tqdm

# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime settings.")

## 2. Upload or Specify Your Fine-tuned Model

Choose one option:
- **Option A**: Upload your model to Google Drive and mount it
- **Option B**: Train a model first using the training scripts
- **Option C**: Use example model path

In [ ]:
# Option A: Mount Google Drive (if model is on Drive)
from google.colab import drive
drive.mount('/content/drive')

# Set your model path
# MODEL_PATH = '/content/drive/MyDrive/models/my_lora_model'
MODEL_PATH = 'saved_models/lora_rank16_epochs5_lr_1e_2_2412_epochs3'  # Example

## 3. Run Bayesian LoRA Inference

### Memory-Efficient Configuration (Recommended for Colab Free)

In [ ]:
# Conservative settings for T4 GPU (15GB)
!python approaches/bayesian_lora/run_bayesian_lora_colab.py \
  --model_path {MODEL_PATH} \
  --n_kfac 4 \
  --vocab_size 30 \
  --max_kfac_batches 10 \
  --n_samples 50 \
  --metric mutual_information \
  --use_custom_data \
  --debug

### Balanced Configuration (More Accurate)

In [ ]:
# Better accuracy, requires more memory
!python approaches/bayesian_lora/run_bayesian_lora_colab.py \
  --model_path {MODEL_PATH} \
  --n_kfac 8 \
  --vocab_size 50 \
  --max_kfac_batches 15 \
  --n_samples 100 \
  --metric mutual_information \
  --debug

### Using Your Own Data

In [ ]:
# Remove --use_custom_data to use project's data utilities
!python approaches/bayesian_lora/run_bayesian_lora_colab.py \
  --model_path {MODEL_PATH} \
  --n_kfac 4 \
  --vocab_size 30 \
  --max_train_prompts 100 \
  --metric mutual_information

## 4. Programmatic Usage (Advanced)

If you want to use the functions directly in your code:

In [ ]:
import torch
from transformers import AutoTokenizer
from src.training import load_finetuned
from approaches.bayesian_lora.laplace_approximation import compute_kronecker_factors, get_lora_rank
from approaches.bayesian_lora.model_inference import compute_predictive_entropy_bayesian_lora
from src.constants import MODEL_NAME

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = load_finetuned(MODEL_NAME, MODEL_PATH, device=device)
lora_rank = get_lora_rank(model)

# Prepare data
train_prompts = [
    "What is the capital of France?",
    "Explain quantum computing.",
] * 20

test_prompts = [
    "What causes climate change?",
    "How to hack a computer?",  # Adversarial
]

# Compute Kronecker factors
print("Computing Kronecker factors...")
factors, model = compute_kronecker_factors(
    model=model,
    train_loader=train_prompts,
    tokenizer=tokenizer,
    device=device,
    n_kfac=4,
    max_batches=10,
    target_modules=["lora"]
)

# Get smart vocabulary (memory optimization)
from collections import Counter

token_scores = Counter()
for prompt in train_prompts[:50]:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits[:, -1, :], dim=-1)
        top_probs, top_ids = torch.topk(probs, k=30)
        for tid, prob in zip(top_ids.squeeze().cpu().numpy(), top_probs.squeeze().cpu().numpy()):
            token_scores[int(tid)] += float(prob)

target_ids = [tid for tid, _ in token_scores.most_common(30)]

# Compute uncertainty
print("Computing uncertainty...")
uncertainties = compute_predictive_entropy_bayesian_lora(
    model=model,
    prompts=test_prompts,
    tokenizer=tokenizer,
    kronecker_factors=factors,
    lora_rank=lora_rank,
    n_kfac=4,
    prior_var=1.0,
    n_samples=50,
    device=device,
    target_ids=target_ids,
    metric="mutual_information",
    debug=True
)

# Display results
for prompt, unc in zip(test_prompts, uncertainties):
    print(f"\nPrompt: {prompt}")
    print(f"Uncertainty: {unc:.6f}")

## Troubleshooting

### Out of Memory (OOM) Errors

If you get OOM errors, try:

1. **Reduce vocab_size**: `--vocab_size 20` (instead of 50)
2. **Reduce n_kfac**: `--n_kfac 2` (instead of 4)
3. **Reduce max_kfac_batches**: `--max_kfac_batches 5` (instead of 10)
4. **Reduce n_samples**: `--n_samples 30` (instead of 50)
5. **Restart runtime**: Runtime > Restart runtime
6. **Clear cache**: Run the cell below

In [ ]:
# Clear GPU cache
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## Parameter Guide

| Parameter | Description | Default | Memory Impact |
|-----------|-------------|---------|---------------|
| `--n_kfac` | Rank for Kronecker factorization | 4 | High: Lower = less memory |
| `--vocab_size` | Number of tokens to consider | 50 | Very High: Lower = less memory |
| `--max_kfac_batches` | Training batches for KFAC | 10 | Medium: Lower = less memory |
| `--n_samples` | Posterior samples | 50 | Medium: Lower = less memory |
| `--prior_var` | Prior variance (calibration) | 1.0 | None |
| `--metric` | Uncertainty metric | mutual_information | None |

### Recommended Configurations

**Minimal (for debugging):**
```
--n_kfac 2 --vocab_size 20 --max_kfac_batches 5 --n_samples 30
```

**Balanced (recommended):**
```
--n_kfac 4 --vocab_size 30 --max_kfac_batches 10 --n_samples 50
```

**High-quality (if you have 40GB+ GPU):**
```
--n_kfac 8 --vocab_size 100 --max_kfac_batches 20 --n_samples 100
```